In [ ]:
import pandas as pd
import requests
from io import StringIO

url = 'http://api.clubelo.com/2025-08-29'

try:
    response = requests.get(url)
    response.raise_for_status()  # Hibát dob, ha a válasz státusza nem 200 OK
    
    # Kiírjuk a válasz tartalmát
    #print("Válasz státusz kód:", response.status_code)
    #print("\nVálasz tartalma:")
    rtext = response.text
    #print(rtext)
    
except requests.exceptions.HTTPError as http_err:
    print(f'HTTP hiba történt: {http_err}')
except requests.exceptions.RequestException as err:
    print(f'Hiba történt a kérés során: {err}')
    
df = pd.read_csv(StringIO(rtext))

print(df.head())

In [ ]:
import math

def calculate_1X2(elo1, elo2, draw_ratio=0.3):
    """
    Realisztikus 1X2 valószínűségek ELO alapján.
    
    Args:
        elo1: Az első csapat ELO-ja
        elo2: A második csapat ELO-ja
        draw_ratio: Max döntetlen esély kiegyenlített csapatoknál (pl. foci ~0.25)
    
    Returns:
        Tuple (prob1, prob_draw, prob2)
    """
    elo_diff = elo1 - elo2
    
    # Két kimenetes győzelmi valószínűség (klasszikus ELO formula)
    prob1_raw = 1 / (1 + math.pow(10, -elo_diff / 400))
    prob2_raw = 1 - prob1_raw
    
    # Döntetlen valószínűsége az ELO különbség függvényében
    # Ha a csapatok közel azonosak, nagyobb az esély
    prob_draw_raw = draw_ratio * math.exp(-abs(elo_diff) / 200)
    
    # Maradék valószínűséget arányosan osztjuk szét a győzelmek között
    remaining = 1 - prob_draw_raw
    prob1 = prob1_raw * remaining
    prob2 = prob2_raw * remaining
    prob_draw = prob_draw_raw
    
    return prob1, prob_draw, prob2

# Példa
p1, pd, p2 = calculate_1X2(1950, 1600)
print(f"1: {p1:.3f}, X: {pd:.3f}, 2: {p2:.3f}")


In [ ]:
team = 'Ferencvaros'
elo_team = df[df.Club == team].Elo.values[0]

opponents = ['Rangers', 'Salzburg', 'Viktoria Plzen', 'Fenerbahce',
             'Razgrad', 'Forest', 'Panathinaikos', 'Genk']
exp_points_all = 0
for opp in opponents:
    elo_opp = df[df.Club == opp].Elo.values[0]

    P_win, P_draw, P_lose = calculate_1X2(elo_team, elo_opp)
    exp_points = P_win*3 + P_draw*1 + P_lose*0
    print(opp, exp_points)
    exp_points_all += exp_points
print(exp_points_all)

In [ ]:
matches_dict = {
    'Roma': ['Lille', 'Rangers', 'Viktoria Plzen', 'Celtic', 'Midtjylland', 'Nice', 'Stuttgart', 'Panathinaikos'],
    'Porto': ['Rangers', 'Salzburg', 'Crvena Zvezda', 'Viktoria Plzen', 'Nice', 'Forest', 'Malmoe', 'Utrecht'],
    'Rangers': ['Roma', 'Porto', 'Braga', 'Ferencvaros', 'Razgrad', 'Sturm Graz', 'Genk', 'Brann'],
    'Feyenoord': ['Aston Villa', 'Betis', 'Celtic', 'Braga', 'Sturm Graz', 'Steaua', 'Panathinaikos', 'Stuttgart'],
    'Lille': ['Dinamo Zagreb', 'Roma', 'PAOK', 'Crvena Zvezda', 'Freiburg', 'Young Boys', 'Brann', 'Celta'],
    'Dinamo Zagreb': ['Betis', 'Lille', 'Fenerbahce', 'M Tel Aviv', 'Steaua', 'Midtjylland', 'Celta', 'Malmoe'],
    'Betis': ['Feyenoord', 'Dinamo Zagreb', 'Lyon', 'PAOK', 'Forest', 'Razgrad', 'Utrecht', 'Genk'],
    'Salzburg': ['Porto', 'Aston Villa', 'Ferencvaros', 'Lyon', 'Basel', 'Freiburg', 'Go Ahead Eagles', 'Bologna'],
    'Aston Villa': ['Salzburg', 'Feyenoord', 'M Tel Aviv', 'Fenerbahce', 'Young Boys', 'Basel', 'Bologna', 'Go Ahead Eagles'],
    'Fenerbahce': ['Aston Villa', 'Dinamo Zagreb', 'Ferencvaros', 'Viktoria Plzen', 'Nice', 'Steaua', 'Stuttgart', 'Brann'],
    'Braga': ['Feyenoord', 'Rangers', 'Crvena Zvezda', 'Celtic', 'Forest', 'Nice', 'Genk', 'Go Ahead Eagles'],
    'Crvena Zvezda': ['Lille', 'Porto', 'Celtic', 'Braga', 'Steaua', 'Sturm Graz', 'Celta', 'Malmoe'],
    'Lyon': ['Salzburg', 'Betis', 'PAOK', 'M Tel Aviv', 'Basel', 'Young Boys', 'Go Ahead Eagles', 'Utrecht'],
    'PAOK': ['Betis', 'Lille', 'M Tel Aviv', 'Lyon', 'Young Boys', 'Razgrad', 'Brann', 'Celta'],
    'Viktoria Plzen': ['Porto', 'Roma', 'Fenerbahce', 'Ferencvaros', 'Freiburg', 'Basel', 'Malmoe', 'Panathinaikos'],
    'Ferencvaros': ['Rangers', 'Salzburg', 'Viktoria Plzen', 'Fenerbahce', 'Razgrad', 'Forest', 'Panathinaikos', 'Genk'],
    'Celtic': ['Roma', 'Feyenoord', 'Braga', 'Crvena Zvezda', 'Sturm Graz', 'Midtjylland', 'Utrecht', 'Bologna'],
    'M Tel Aviv': ['Dinamo Zagreb', 'Aston Villa', 'Lyon', 'PAOK', 'Midtjylland', 'Freiburg', 'Bologna', 'Stuttgart'],
    'Young Boys': ['Lille', 'Aston Villa', 'Lyon', 'PAOK', 'Razgrad', 'Steaua', 'Panathinaikos', 'Stuttgart'],
    'Basel': ['Aston Villa', 'Salzburg', 'Viktoria Plzen', 'Lyon', 'Steaua', 'Freiburg', 'Stuttgart', 'Genk'],
    'Midtjylland': ['Dinamo Zagreb', 'Roma', 'Celtic', 'M Tel Aviv', 'Sturm Graz', 'Forest', 'Genk', 'Brann'],
    'Freiburg': ['Salzburg', 'Lille', 'M Tel Aviv', 'Viktoria Plzen', 'Basel', 'Nice', 'Utrecht', 'Bologna'],
    'Razgrad': ['Betis', 'Rangers', 'PAOK', 'Ferencvaros', 'Nice', 'Young Boys', 'Celta', 'Malmoe'],
    'Forest': ['Porto', 'Betis', 'Ferencvaros', 'Braga', 'Midtjylland', 'Sturm Graz', 'Malmoe', 'Utrecht'],
    'Sturm Graz': ['Rangers', 'Feyenoord', 'Crvena Zvezda', 'Celtic', 'Forest', 'Midtjylland', 'Brann', 'Panathinaikos'],
    'Steaua': ['Feyenoord', 'Dinamo Zagreb', 'Fenerbahce', 'Crvena Zvezda', 'Young Boys', 'Basel', 'Bologna', 'Go Ahead Eagles'],
    'Nice': ['Roma', 'Porto', 'Braga', 'Fenerbahce', 'Freiburg', 'Razgrad', 'Go Ahead Eagles', 'Celta'],
    'Bologna': ['Salzburg', 'Aston Villa', 'Celtic', 'M Tel Aviv', 'Freiburg', 'Steaua', 'Brann', 'Celta'],
    'Celta': ['Lille', 'Dinamo Zagreb', 'PAOK', 'Crvena Zvezda', 'Nice', 'Razgrad', 'Bologna', 'Stuttgart'],
    'Stuttgart': ['Feyenoord', 'Roma', 'M Tel Aviv', 'Fenerbahce', 'Young Boys', 'Basel', 'Celta', 'Go Ahead Eagles'],
    'Panathinaikos': ['Roma', 'Feyenoord', 'Viktoria Plzen', 'Ferencvaros', 'Sturm Graz', 'Young Boys', 'Go Ahead Eagles', 'Malmoe'],
    'Malmoe': ['Dinamo Zagreb', 'Porto', 'Crvena Zvezda', 'Viktoria Plzen', 'Razgrad', 'Forest', 'Panathinaikos', 'Genk'],
    'Go Ahead Eagles': ['Aston Villa', 'Salzburg', 'Braga', 'Lyon', 'Steaua', 'Nice', 'Stuttgart', 'Panathinaikos'],
    'Utrecht': ['Porto', 'Betis', 'Lyon', 'Celtic', 'Forest', 'Freiburg', 'Genk', 'Brann'],
    'Genk': ['Betis', 'Rangers', 'Ferencvaros', 'Braga', 'Basel', 'Midtjylland', 'Malmoe', 'Utrecht'],
    'Brann': ['Rangers', 'Lille', 'Fenerbahce', 'PAOK', 'Midtjylland', 'Sturm Graz', 'Utrecht', 'Bologna']
}

for team in matches_dict.keys():
    if team not in df.Club.values:
        print(f'{team} not found')

In [ ]:
import pandas as pd
import math
import random

# Hozzon létre egy listát az összes egyedi mérkőzésről
all_matches = []
processed_pairs = set()

for team, opponents in matches_dict.items():
    # Feltételezve, hogy a lista első 4 ellenfele hazai mérkőzés
    for i in range(4):
        home_team = team
        away_team = opponents[i]
        match_tuple = tuple(sorted([home_team, away_team]))
        if match_tuple not in processed_pairs:
            all_matches.append({'Home': home_team, 'Away': away_team})
            processed_pairs.add(match_tuple)
    
    # És a második 4 ellenfele idegenbeli mérkőzés
    for i in range(4, 8):
        away_team = team
        home_team = opponents[i]
        match_tuple = tuple(sorted([home_team, away_team]))
        if match_tuple not in processed_pairs:
            all_matches.append({'Home': home_team, 'Away': away_team})
            processed_pairs.add(match_tuple)

# Monte-Carlo szimuláció
NUM_SIMULATIONS = 10000
team_outcomes = {team: {'top_8': 0, '9_24': 0, 'eliminated': 0} for team in matches_dict.keys()}

elo_map = df.set_index('Club')['Elo'].to_dict()

for _ in range(NUM_SIMULATIONS):
    current_points = {team: 0 for team in matches_dict.keys()}
    
    for match in all_matches:
        home_team = match['Home']
        away_team = match['Away']
        
        elo_home = elo_map.get(home_team)
        elo_away = elo_map.get(away_team)
        
        # A valószínűségek kiszámítása a meglévő függvény alapján
        if elo_home and elo_away:
            P_home_win, P_draw, P_away_win = calculate_1X2(elo_home, elo_away)
            
            # Véletlen generálás a győztes meghatározásához
            rand_num = random.uniform(0, 1)
            
            if rand_num < P_home_win:
                current_points[home_team] += 3
            elif rand_num < P_home_win + P_draw:
                current_points[home_team] += 1
                current_points[away_team] += 1
            else:
                current_points[away_team] += 3
    
    # Az eredmények kiértékelése egy szimuláció után
    sorted_teams = sorted(current_points.items(), key=lambda item: item[1], reverse=True)
    
    for i, (team, points) in enumerate(sorted_teams):
        if i < 8:
            team_outcomes[team]['top_8'] += 1
        elif i < 24:
            team_outcomes[team]['9_24'] += 1
        else:
            team_outcomes[team]['eliminated'] += 1

# Végső eredmények kiírása
for team, outcomes in sorted(team_outcomes.items(), key=lambda item: item[1]['top_8'], reverse=True):
    print(f"\nCsapat: {team}")
    print(f"  Bejutás a legjobb 8-ba: {(outcomes['top_8'] / NUM_SIMULATIONS) * 100:.2f}%")
    print(f"  Bejutás a legjobb 9-24-be: {(outcomes['9_24'] / NUM_SIMULATIONS) * 100:.2f}%")
    print(f"  Kiesés (25-36. hely): {(outcomes['eliminated'] / NUM_SIMULATIONS) * 100:.2f}%")

In [ ]:
# Tabella vizualizáció
# Konvertálás DataFrame-mé és rendezés
table_df = pd.DataFrame(expected_points.items(), columns=['Csapat', 'Várható Pontszám'])
table_df['Várható Pontszám'] = table_df['Várható Pontszám'].round(2)
table_df.sort_values(by='Várható Pontszám', ascending=False, inplace=True)
table_df.reset_index(drop=True, inplace=True)
table_df.index += 1
table_df.rename_axis('Helyezés', inplace=True)
display(table_df)

In [ ]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.offline as pyo

simulation_results = {team: [] for team in matches_dict.keys()}
# Hozzáadom a korábban megadott szimulációs kódot
# A szimuláció futtatása a továbbjutási esélyek kiszámításához
NUM_SIMULATIONS = 10000
team_outcomes = {team: {'top_8': 0, '9_24': 0, 'eliminated': 0} for team in matches_dict.keys()}
elo_map = df.set_index('Club')['Elo'].to_dict()
all_matches = []
processed_pairs = set()
for team, opponents in matches_dict.items():
    for i in range(4):
        home_team = team
        away_team = opponents[i]
        match_tuple = tuple(sorted([home_team, away_team]))
        if match_tuple not in processed_pairs:
            all_matches.append({'Home': home_team, 'Away': away_team})
            processed_pairs.add(match_tuple)
    for i in range(4, 8):
        away_team = team
        home_team = opponents[i]
        match_tuple = tuple(sorted([home_team, away_team]))
        if match_tuple not in processed_pairs:
            all_matches.append({'Home': home_team, 'Away': away_team})
            processed_pairs.add(match_tuple)
for _ in range(NUM_SIMULATIONS):
    current_points = {team: 0 for team in matches_dict.keys()}
    for match in all_matches:
        home_team = match['Home']
        away_team = match['Away']
        elo_home = elo_map.get(home_team)
        elo_away = elo_map.get(away_team)
        if elo_home and elo_away:
            P_home_win, P_draw, P_away_win = calculate_1X2(elo_home, elo_away)
            rand_num = random.uniform(0, 1)
            if rand_num < P_home_win:
                current_points[home_team] += 3
            elif rand_num < P_home_win + P_draw:
                current_points[home_team] += 1
                current_points[away_team] += 1
            else:
                current_points[away_team] += 3
    sorted_teams = sorted(current_points.items(), key=lambda item: item[1], reverse=True)
    for i, (team, points) in enumerate(sorted_teams):
        if i < 8:
            team_outcomes[team]['top_8'] += 1
        elif i < 24:
            team_outcomes[team]['9_24'] += 1
        else:
            team_outcomes[team]['eliminated'] += 1

# A kód a vizualizációhoz
background_color = '#3c3d3d'
mycolor = '#5ECB43'

# Kiválasztott csapat
team_name = 'Aston Villa'

# Esélyek százalékos értékekké konvertálása
top_8_perc = (team_outcomes[team_name]['top_8'] / NUM_SIMULATIONS) * 100
_9_24_perc = (team_outcomes[team_name]['9_24'] / NUM_SIMULATIONS) * 100
elim_perc = (team_outcomes[team_name]['eliminated'] / NUM_SIMULATIONS) * 100

labels = [team_name]
top_8_values = [top_8_perc]
_9_24_values = [_9_24_perc]
elim_values = [elim_perc]
width = 0.5

# Diagram létrehozása
fig, ax = plt.subplots(figsize=(6, 8), facecolor=background_color)
ax.set_facecolor(background_color)

# Oszlopok felhalmozása a kívánt színekkel
ax.bar(labels, top_8_values, width, label='Top 8', color=mycolor)
ax.bar(labels, _9_24_values, width, bottom=top_8_values, label='9-24th place', color='lightgreen') # Kicsit eltérő zöld
ax.bar(labels, elim_values, width, bottom=[t + e for t, e in zip(top_8_values, _9_24_values)], label='Elimination', color='red') # Piros a kiesésre

# Címkék és feliratok a kért fehér színnel
ax.set_ylabel('Probability (%)', color='white')
ax.set_title(f'{team_name} chances in UEL', color='white')
ax.legend(facecolor=background_color, edgecolor='white', labelcolor='white')

# Tengelyek stílusa
ax.spines['bottom'].set_color('white')
ax.spines['top'].set_visible(False)
ax.spines['left'].set_color('white')
ax.spines['right'].set_visible(False)
ax.tick_params(colors='white')
plt.grid(True, axis='y', alpha=0.3, color='white')

# Pontos százalékok hozzáadása az oszlopokhoz fehér felirattal
y_offset = -4
ax.text(0, top_8_perc/2 + y_offset, f'{top_8_perc:.2f}%', ha='center', color='white', fontweight='bold')
ax.text(0, top_8_perc + _9_24_perc/2 + y_offset, f'{_9_24_perc:.2f}%', ha='center', color='white', fontweight='bold')
#ax.text(0, top_8_perc + _9_24_perc + elim_perc/2 + y_offset, f'{elim_perc:.2f}%', ha='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns 
import pandas as pd

# A kód ezen része feltételezi, hogy a 'simulation_results' szótár már
# ki van számolva a korábbi cellák futtatásával.
simulation_results = {team: [] for team in matches_dict.keys()}
for _ in range(NUM_SIMULATIONS):
    current_points = {team: 0 for team in matches_dict.keys()}
    for match in all_matches:
        home_team = match['Home']
        away_team = match['Away']
        elo_home = elo_map.get(home_team)
        elo_away = elo_map.get(away_team)
        if elo_home and elo_away:
            P_home_win, P_draw, P_away_win = calculate_1X2(elo_home, elo_away)
            rand_num = random.uniform(0, 1)
            if rand_num < P_home_win:
                current_points[home_team] += 3
            elif rand_num < P_home_win + P_draw:
                current_points[home_team] += 1
                current_points[away_team] += 1
            else:
                current_points[away_team] += 3
    sorted_teams = sorted(current_points.items(), key=lambda item: item[1], reverse=True)
    for i, (team, points) in enumerate(sorted_teams):
        if i < 8:
            team_outcomes[team]['top_8'] += 1
        elif i < 24:
            team_outcomes[team]['9_24'] += 1
        else:
            team_outcomes[team]['eliminated'] += 1
        simulation_results[team].append(i + 1)
# A kód a vizualizációhoz
background_color = '#3c3d3d'
mycolor = '#5ECB43'

selected_teams = ['Ferencvaros', 'Aston Villa', 'Forest', 'Salzburg']
simulation_df = pd.DataFrame(simulation_results)
filtered_df = simulation_df[selected_teams]

plt.figure(figsize=(12, 8), facecolor=background_color)
ax = plt.gca()
ax.set_facecolor(background_color)

# Hegedűdiagram a kért zöld színnel
sns.violinplot(data=filtered_df, palette=[mycolor] * len(selected_teams), inner='quartile', ax=ax, saturation=1)
ax.set_xlabel('Team', color='white')
ax.set_ylabel('Place', color='white')
ax.set_title('Europa League Placement Distribution', color='white')
ax.grid(True, axis='y', color='white', alpha=0.3)
ax.set_ylim(1, 36)

# Tengelyek és feliratok stílusa
ax.spines['bottom'].set_color('white')
ax.spines['top'].set_visible(False)
ax.spines['left'].set_color('white')
ax.spines['right'].set_visible(False)
ax.tick_params(colors='white')

plt.tight_layout()
plt.show()